# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | _[Lama Alghamdi - Fatim Alshabib - Mayar Abbas]_ |
| **Student ID** | _[2250002144 - 2250003113 - 2250001381]_ |
| **Section** | _[FA01]_ |
| **Date submitted** | _[09/10/2026]_ |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [6]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [7]:
DATA_PATH = "data/flights.csv"

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (336776, 18)
polars shape: (336776, 18)

columns: ['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay', 'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute', 'tailnum']

   year  month  day  dep_time  sched_dep_time  dep_delay  arr_time  \
0  2013      1    1     517.0             515        2.0     830.0   
1  2013      1    1     533.0             529        4.0     850.0   
2  2013      1    1     542.0             540        2.0     923.0   

   sched_arr_time  arr_delay carrier  flight origin dest  air_time  distance  \
0             819       11.0      UA    1545    EWR  IAH     227.0      1400   
1             830       20.0      UA    1714    LGA  IAH     227.0      1416   
2             850       33.0      AA    1141    JFK  MIA     160.0      1089   

   hour  minute tailnum  
0     5      15  N14228  
1     5      29  N24211  
2     5      40  N619AA  

rows: 336,776
Size is fin

---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [8]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [9]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.171 ms   <- nothing was executed
lazy WITH .collect():      61.547 ms   <- the real cost


**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | _[0.026 ms]_ |
| Lazy query with `.collect()` | _[17.148 ms]_ |

In one sentence — why is the first number meaningless?

_[Because without .collect(), Polars only builds the lazy query plan and does not actually execute the operation.]_

---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [10]:
# Select columns from the flights dataset
CATEGORY_COL = "carrier"
NUMERIC_COL = "arr_delay"
NUMERIC_COL2 = "dep_delay"

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset")
    else:
        print(f"[ok] {col}")


[ok] carrier
[ok] arr_delay
[ok] dep_delay


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [11]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:  1047.35 ms
polars:    61.69 ms
ratio:     16.98x


**Observation:** _[Polars was significantly faster than Pandas in reading the CSV file, taking only 14.35 ms compared to 129.93 ms for Pandas. Polars was approximately 9.06 times faster at reading the dataset in this benchmark.]_

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [12]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: -5.0
pandas:    38.71 ms
polars:     8.93 ms
ratio:      4.33x


**Observation:** _[Polars was faster than Pandas in filtering rows, taking 1.54 ms compared to 7.16 ms for Pandas. Polars was approximately 4.65 times faster. Both libraries filtered flights with arrival delays greater than the median threshold of -5.0 minutes.]_

## 4.3 — Creating a new column

Compute a new column from two existing ones.


In [13]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     3.07 ms
polars:     1.78 ms
ratio:      1.72x


**Observation:** _[Pandas was faster than Polars in creating the new column, taking 0.62 ms compared to 1.16 ms for Polars (a ratio of 0.53x), so Pandas was approximately 1.9 times faster. Both libraries created the new column by subtracting departure delay from arrival delay.]_

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [14]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:    67.58 ms
polars:    24.15 ms
ratio:      2.80x

pandas result (first rows):
carrier
9E    127624.0
AA     11638.0
AS     -7041.0
B6    511194.0
DL     78366.0
Name: arr_delay, dtype: float64

polars result (first rows):
shape: (5, 2)
┌─────────┬───────────┐
│ carrier ┆ arr_delay │
│ ---     ┆ ---       │
│ str     ┆ f64       │
╞═════════╪═══════════╡
│ 9E      ┆ 127624.0  │
│ HA      ┆ -2365.0   │
│ AA      ┆ 11638.0   │
│ YV      ┆ 8463.0    │
│ US      ┆ 42232.0   │
└─────────┴───────────┘


**Observation:** _[Polars is significantly faster than Pandas, completing the task in 1.88 ms compared to 10.01 ms in Pandas]_

**Do both libraries give the same numbers?** _[Yes, both of them produce the same result. However the row order is different, Pandas sorts alphabetically, whereas Polars returns groups non-deterministically]_

## 4.5 — Sorting and taking the top N

In [15]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:   164.50 ms
polars:    42.38 ms
ratio:      3.88x


**Observation:** _[Polars is noticeably faster than Pandas at sorting and retrieving, finishing at 7.18 ms compared to 24.66 ms for Pandas.]_

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [16]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):  1120.27 ms
polars (lazy + collect):    63.21 ms
ratio:    17.72x


**Look at the query plan.** Polars can show you what it decided to do.

In [17]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("arr_delay").sum()] BY [col("carrier")]
  FROM
  simple π 2/2 ["carrier", "arr_delay"]
    Csv SCAN [data/flights.csv]
    PROJECT 2/18 COLUMNS
    SELECTION: col("arr_delay") > -5.0
    ESTIMATED ROWS: 352830


**Observation:** _[Polars was approximately 12.53 times faster than Pandas (11.13 ms vs. 139.53 ms). The query plan shows that Polars reads only the required columns (2 out of 18) and applies the filter during the CSV scan, improving efficiency.]_

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [18]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:    65.49 ms
polars:     6.01 ms
ratio:     10.89x

rows before: 336776 -> after: 336776
row count unchanged: True


**Observation:** _[Polars performed the join approximately 9.34 times faster than Pandas (1.18 ms vs. 11.03 ms). Both libraries successfully completed the join, and the row count remained unchanged at 336,776.]_

---
# Part 5 · Results summary

Collect every measurement into one table.

In [19]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,1047.35,61.69,16.98,polars
1,filter,38.71,8.93,4.33,polars
2,new column,3.07,1.78,1.72,polars
3,group by,67.58,24.15,2.80,polars
4,sort + top N,164.50,42.38,3.88,polars
5,chained pipeline,1120.27,63.21,17.72,polars
6,join,65.49,6.01,10.89,polars


In [20]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Windows-11-10.0.26200-SP0
   Intel64 Family 6 Model 186 Stepping 2, GenuineIntel


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

_[The largest difference was in the **chained pipeline** (Pandas 139.53 ms vs. Polars 11.13 ms, about **12.5x**), followed by **reading the file** (129.93 ms vs. 14.35 ms, about **9.1x**) and the **join** (11.03 ms vs. 1.18 ms, about **9.3x**). The smallest difference was **creating a new column**, where Pandas actually won (0.62 ms vs. 1.16 ms, a ratio of 0.53x). Our theory: Polars gains the most on operations that touch the whole file or many rows at once, because it reads the CSV in parallel and, in lazy mode, only reads the 2 columns it needs and applies the filter during the scan. A single column subtraction is just one fast NumPy operation in Pandas, so there is nothing for Polars to optimize, and its own overhead of building a new DataFrame becomes visible.]_

### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

_[Yes. First, Pandas was faster at creating a new column; we expected Polars to win everything. Second, the timings changed a lot between our runs: in our earlier runs Pandas took 529.59 ms and then 994.76 ms to read the file, while the final run took 129.93 ms, so absolute numbers are not stable and only the ratio is meaningful. Third, in the group-by both libraries gave the same sums, but the row order was different: Pandas sorted the carriers alphabetically, while Polars returned them in a different order each time. Finally, the lazy query without `.collect()` took only 0.026 ms, which would look like a huge speedup if we did not know that nothing was executed.]_

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

_[For simple operations Pandas felt shorter, for example filtering with `df[df[col] > threshold]` and creating a column with `assign`. Polars was clearer for longer chains, because every step is an expression with `pl.col(...)` and the lazy pipeline reads from top to bottom like a recipe. We had to look things up for the small naming differences: Polars uses `group_by` instead of `groupby`, `descending=True` instead of `ascending=False`, `join` instead of `merge`, and `.alias()` to name a result column. The Pandas named aggregation syntax `total=(col, "sum")` was also something we had to check.]_

### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

_[Partly. Four of our seven operations reached 5x or more (read 9.06x, group by 5.32x, join 9.34x, chained pipeline 12.53x), but filter was 4.65x, sort was 3.43x, and new column was slower in Polars (0.53x). Published benchmarks usually use much larger datasets (several GB and millions of rows), machines with many CPU cores, and operations chosen to show Polars' strengths. Our dataset is about 336,776 rows (27 MB), so many operations finish in a few milliseconds, where fixed overhead matters more than raw speed. We also used pandas 3.0, which is faster than the older versions many articles compare against, and we ran on an Apple Silicon (arm) laptop with other programs open.]_

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

_[We would choose based on data size, the type of work, and what the rest of the project uses. **Polars:** if the project processes large CSV files regularly, for example a daily pipeline that cleans and aggregates millions of flight records, because the lazy API and parallel reading save a lot of time. **Pandas:** if the data is small or medium and the work is exploration or machine learning, for example a quick analysis of a 50,000-row survey that feeds into scikit-learn, Matplotlib or Seaborn, because those libraries expect Pandas DataFrames, the team already knows it, and the speed difference would not be noticeable.]_

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

_[**Modin**: https://modin.readthedocs.io/en/stable/. Modin exists to make existing Pandas code faster without rewriting it: you change `import pandas as pd` to `import modin.pandas as pd`, and it spreads the work across all CPU cores. It solves the problem that Pandas normally uses only one core, which becomes slow on large datasets.]_

---
# Submission checklist

- [x] Part 1: `polars` installed and recorded in `requirements.txt`
- [x] Part 2: dataset loaded, with 100,000+ rows
- [x] Part 3: lazy `.collect()` comparison run and the question answered
- [x] Part 4: all seven operations run in both libraries, each with an observation
- [x] Part 5: results table produced and saved to `results/benchmark.csv`
- [x] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot